# Forecast Global de Ativos Financeiros Usando Múltiplos Modelos de Deep Learning para Séries Temporais

## Carga das Bibliotecas

In [1]:
################################################################################################################################
########## Importando Bibliotecas Básicas ######################################################################################
################################################################################################################################
import sys
import gc
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import time
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch

In [3]:
from utilsforecast.losses import *
from utilsforecast.processing import *
from utilsforecast.evaluation import *
from utilsforecast.feature_engineering import *

In [4]:
from functools import partial
from functools import reduce

In [5]:
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss, DistributionLoss, HuberMQLoss, MAE, SMAPE, PMM, MSE, GMM, RMSE
from neuralforecast.models import NBEATSx, NHITS, DeepAR, NLinear, LSTM, RNN, DLinear, BiTCN, TFT
from neuralforecast.utils import PredictionIntervals

In [6]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch

In [7]:
from typing import Tuple,List, Dict
from datetime import timedelta

In [8]:
import logging
import ray

logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)
ray.init(log_to_driver=False)

2026-06-23 13:29:29,881	INFO worker.py:2012 -- Started a local Ray instance.


Python version:,3.11.15
Ray version:,2.55.1


In [9]:
import os

os.environ['NIXTLA_ID_AS_COL'] = '1'
# don't reset the index on the output from predict if that's what you're currently doing

In [10]:
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger

# Disable logging
trainer_no_log = pl.Trainer(logger=False)

# Use a specific logger
csv_logger = CSVLogger("logs", name="my_experiment")
trainer_custom_log = pl.Trainer(logger=csv_logger)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [11]:
# Detecta o melhor acelerador disponível: CUDA, ROCm, MPS ou CPU
def _detectar_acelerador():
    # CUDA (NVIDIA) e ROCm (AMD) usam a mesma API torch.cuda no PyTorch
    if torch.cuda.is_available():
        return 'gpu', [0]
    # MPS (Apple Silicon / Metal)
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return 'mps', 1
    # Fallback para CPU
    return 'cpu', 1

_accelerator, _devices = _detectar_acelerador()

# Argumentos do treinador para os modelos da NeuralForecast
TRAINER_KWARGS = {
    'accelerator': _accelerator,
    'devices': _devices,
}

## Funções Básicas de Processamento

In [12]:
def split_data(
    df: pd.DataFrame,
    horizon: int,
    mode: str
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide o DataFrame em treino, validação e teste."""
    end_date = df['ds'].max()
    test_start_date = end_date - timedelta(days=horizon - 1)
    
    if mode == "treino":
        train_df = df.copy()
        validation_df = pd.DataFrame()
        test_df = pd.DataFrame()
        print(f"Modo Treino: Usando todos os dados de {df['ds'].min():%Y-%m-%d} a {end_date:%Y-%m-%d}")
        
    elif mode == "validacao":
        validation_start_date = test_start_date - timedelta(days=horizon)
        train_end_date = validation_start_date - timedelta(days=1)
        
        train_df = df[df['ds'] <= train_end_date]
        validation_df = df[(df['ds'] >= validation_start_date) & (df['ds'] < test_start_date)]
        test_df = df[df['ds'] >= test_start_date]
        
    elif mode == "backtest":
        train_end_date = test_start_date - timedelta(days=1)
        train_df = df[df['ds'] <= train_end_date]
        test_df = df[df['ds'] >= test_start_date]
        validation_df = pd.DataFrame()
        
    else:
        raise ValueError(f"Modo '{mode}' não reconhecido.")

    return train_df, validation_df, test_df

In [13]:


#from functools import partial
#import numpy as np
#import pandas as pd
#from mlforecast.feature_engineering import pipeline
#from utilsforecast.feature_engineering import trend, fourier


# ---------------------------------------------------------------------------
# Calendário de mercado
# ---------------------------------------------------------------------------

def _easter(year):
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19*a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    L = (32 + 2*e + 2*i - h - k) % 7
    m = (a + 11*h + 22*L) // 451
    month = (h + L - 7*m + 114) // 31
    day = ((h + L - 7*m + 114) % 31) + 1
    return pd.Timestamp(year=year, month=month, day=day)

def b3_holidays(years):
    rows = []
    for y in years:
        easter = _easter(y)
        rows += [
            ("ano_novo",          pd.Timestamp(y, 1, 1)),
            ("carnaval_seg",      easter - pd.Timedelta(days=48)),
            ("carnaval_ter",      easter - pd.Timedelta(days=47)),
            ("sexta_santa",       easter - pd.Timedelta(days=2)),
            ("tiradentes",        pd.Timestamp(y, 4, 21)),
            ("dia_trabalho",      pd.Timestamp(y, 5, 1)),
            ("corpus_christi",    easter + pd.Timedelta(days=60)),
            ("independencia",     pd.Timestamp(y, 9, 7)),
            ("aparecida",         pd.Timestamp(y, 10, 12)),
            ("finados",           pd.Timestamp(y, 11, 2)),
            ("republica",         pd.Timestamp(y, 11, 15)),
            ("consciencia_negra", pd.Timestamp(y, 11, 20)),
            ("natal",             pd.Timestamp(y, 12, 25)),
        ]
    return pd.DataFrame(rows, columns=["event", "date"])

def nyse_holidays(years):
    def nth_weekday(y, m, weekday, n):
        d = pd.Timestamp(y, m, 1)
        offset = (weekday - d.weekday()) % 7 + 7 * (n - 1)
        return d + pd.Timedelta(days=offset)
    def last_weekday(y, m, weekday):
        d = pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0)
        return d - pd.Timedelta(days=(d.weekday() - weekday) % 7)

    rows = []
    for y in years:
        easter = _easter(y)
        rows += [
            ("us_ano_novo",     pd.Timestamp(y, 1, 1)),
            ("us_mlk",          nth_weekday(y, 1, 0, 3)),
            ("us_presidents",   nth_weekday(y, 2, 0, 3)),
            ("us_good_friday",  easter - pd.Timedelta(days=2)),
            ("us_memorial",     last_weekday(y, 5, 0)),
            ("us_juneteenth",   pd.Timestamp(y, 6, 19)),
            ("us_july4",        pd.Timestamp(y, 7, 4)),
            ("us_labor",        nth_weekday(y, 9, 0, 1)),
            ("us_thanksgiving", nth_weekday(y, 11, 3, 4)),
            ("us_christmas",    pd.Timestamp(y, 12, 25)),
        ]
    return pd.DataFrame(rows, columns=["event", "date"])


# ---------------------------------------------------------------------------
# Eventos macro
# ---------------------------------------------------------------------------

def copom_schedule(copom_dates):
    return pd.DataFrame({"event": "copom", "date": pd.to_datetime(copom_dates)})

def fomc_schedule(fomc_dates):
    return pd.DataFrame({"event": "fomc", "date": pd.to_datetime(fomc_dates)})

def ipca_release_dates(years):
    return pd.DataFrame({
        "event": "ipca",
        "date": [pd.Timestamp(y, m, 10) for y in years for m in range(1, 13)],
    })

def nfp_release_dates(years):
    dates = []
    for y in years:
        for m in range(1, 13):
            d = pd.Timestamp(y, m, 1)
            d += pd.Timedelta(days=(4 - d.weekday()) % 7)
            dates.append(d)
    return pd.DataFrame({"event": "nfp", "date": dates})


# ---------------------------------------------------------------------------
# Transformações de DataFrame (operam fora do pipeline)
# ---------------------------------------------------------------------------

def add_calendar_basics(df):
    ds = pd.to_datetime(df["ds"])
    df = df.copy()
    df["dayofweek"] = ds.dt.dayofweek.astype("int8")
    df["day"]       = ds.dt.day.astype("int8")
    df["month"]     = ds.dt.month.astype("int8")
    df["quarter"]   = ds.dt.quarter.astype("int8")
    df["week"]      = ds.dt.isocalendar().week.astype("int16")
    df["dayofyear"] = ds.dt.dayofyear.astype("int16")
    df["year"]      = ds.dt.year.astype("int16")
    return df

def add_event_proximity(df, calendar, prefix="", windows=(-3, -2, -1, 0, 1, 2)):
    if len(calendar) == 0:
        return df
    df = df.copy()
    dates = pd.to_datetime(df["ds"]).values
    for event in calendar["event"].unique():
        event_dates = calendar.loc[calendar["event"] == event, "date"].values
        if len(event_dates) == 0:
            continue
        diffs = (dates[:, None] - event_dates[None, :]).astype("timedelta64[D]").astype(int)
        nearest = diffs[np.arange(len(diffs)), np.abs(diffs).argmin(axis=1)]
        for w in windows:
            df[f"{prefix}is_{event}_d{w:+d}"] = (nearest == w).astype("int8")
    return df

def add_turn_of_month(df, pre=3, post=3, id_col="unique_id"):
    df = df.copy().reset_index(drop=True)
    ds = pd.to_datetime(df["ds"])
    period = ds.dt.to_period("M")
    grp = [df[id_col], period] if id_col in df.columns else [period]
    bdom     = ds.groupby(grp).transform(lambda s: np.arange(1, len(s) + 1))
    bdom_rev = ds.groupby(grp).transform(lambda s: np.arange(len(s), 0, -1))
    df["bday_of_month"]     = bdom.astype("int8").values
    df["bday_to_month_end"] = bdom_rev.astype("int8").values
    df["is_turn_of_month"]  = ((bdom <= post) | (bdom_rev <= pre)).astype("int8").values
    df["is_last_bday"]      = (bdom_rev == 1).astype("int8").values
    df["is_first_bday"]     = (bdom == 1).astype("int8").values
    return df
    
def add_options_expiry(df):
    df = df.copy()
    ds = pd.to_datetime(df["ds"])
    is_friday = ds.dt.dayofweek == 4
    is_third_week = ds.dt.day.between(15, 21)
    df["is_opex"] = (is_friday & is_third_week).astype("int8")
    df["is_triple_witching"] = (
        df["is_opex"].astype(bool) & ds.dt.month.isin([3, 6, 9, 12])
    ).astype("int8")
    return df

def add_calendar_anomalies(df):
    df = df.copy()
    ds = pd.to_datetime(df["ds"])
    df["is_january"]     = (ds.dt.month == 1).astype("int8")
    df["is_sell_in_may"] = ds.dt.month.isin([5, 6, 7, 8, 9, 10]).astype("int8")
    df["is_q4_taxloss"]  = ds.dt.month.isin([10, 11, 12]).astype("int8")
    dec_end   = (ds.dt.month == 12) & (ds.dt.day >= 24)
    jan_start = (ds.dt.month == 1)  & (ds.dt.day <= 3)
    df["is_santa_rally"] = (dec_end | jan_start).astype("int8")
    df["is_quarter_end"] = ds.dt.is_quarter_end.astype("int8")
    df["is_year_end"]    = ds.dt.is_year_end.astype("int8")
    return df

def add_dow_flags(df):
    df = df.copy()
    dow = pd.to_datetime(df["ds"]).dt.dayofweek
    df["is_monday"] = (dow == 0).astype("int8")
    df["is_friday"] = (dow == 4).astype("int8")
    return df


# ---------------------------------------------------------------------------
# Pipeline principal
# ---------------------------------------------------------------------------

def finance_feature_transform(
    df_train,
    freq,
    horizon,
    market="br",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True,
):
    """
    Decomposição para séries diárias (business days) de ativos financeiros.

    Fase 1 (utilsforecast.pipeline): trend + Fourier (semanal/anual/mensal).
    Fase 2 (pandas direto): calendário, feriados, eventos macro, turn-of-month,
                            opex, anomalias, flags de DOW.
    """
    # ------------------------ calendários ----------------------------------
    years_in_data = pd.to_datetime(df_train["ds"]).dt.year
    all_years = list(range(int(years_in_data.min()) - 1,
                           int(years_in_data.max()) + 2))

    holiday_dfs = []
    if market in ("br", "both"):
        holiday_dfs.append(b3_holidays(all_years))
    if market in ("us", "both"):
        holiday_dfs.append(nyse_holidays(all_years))
    holiday_cal = (pd.concat(holiday_dfs, ignore_index=True)
                   if holiday_dfs else pd.DataFrame(columns=["event", "date"]))

    macro_parts = []
    if market in ("br", "both"):
        macro_parts.append(ipca_release_dates(all_years))
        if copom_dates is not None:
            macro_parts.append(copom_schedule(copom_dates))
    if market in ("us", "both"):
        macro_parts.append(nfp_release_dates(all_years))
        if fomc_dates is not None:
            macro_parts.append(fomc_schedule(fomc_dates))
    macro_cal = (pd.concat(macro_parts, ignore_index=True)
                 if macro_parts else pd.DataFrame(columns=["event", "date"]))

    # ------------------------ Fase 1: pipeline -----------------------------
    builtin_features = []
    if include_trend:
        builtin_features.append(trend)
    builtin_features += [
        partial(fourier, season_length=5,   k=2),
        partial(fourier, season_length=252, k=8),
        partial(fourier, season_length=21,  k=2),
    ]
    transformed_df, future_df = pipeline(
        df_train,
        features=builtin_features,
        freq=freq,
        h=horizon,
    )

    # ------------------------ Fase 2: pandas direto ------------------------
    def enrich(df):
        df = add_calendar_basics(df)
        df = add_event_proximity(df, holiday_cal, prefix="hol_")
        df = add_event_proximity(df, macro_cal,   prefix="mac_",
                                 windows=(-2, -1, 0, 1, 2))
        df = add_turn_of_month(df)
        df = add_options_expiry(df)
        df = add_calendar_anomalies(df)
        df = add_dow_flags(df)
        return df

    transformed_df = enrich(transformed_df)
    future_df      = enrich(future_df)

    return transformed_df, future_df

In [14]:
Y_df_ret = pd.read_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_portfolio.parquet')
Y_df_ret

,unique_id,ds,y
0,3m,2022-01-03,0.000387
1,3m,2022-01-04,0.013912
2,3m,2022-01-05,-0.004114
3,3m,2022-01-06,-0.008336
4,3m,2022-01-07,0.010896
...,...,...,...
276279,xp,2026-06-12,0.019133
276280,xp,2026-06-15,-0.026815
276281,xp,2026-06-16,0.018287
276282,xp,2026-06-17,-0.012112


In [15]:
Y_df_vol = pd.read_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_vol_portfolio.parquet')
Y_df_vol

,unique_id,ds,y
0,3m,2022-01-03,55.284016
1,3m,2022-01-04,55.284016
2,3m,2022-01-05,55.284016
3,3m,2022-01-06,55.284016
4,3m,2022-01-07,55.284016
...,...,...,...
276279,xp,2026-06-12,67.648301
276280,xp,2026-06-15,67.648301
276281,xp,2026-06-16,67.648301
276282,xp,2026-06-17,67.648301


In [16]:
Y_df = Y_df_ret.merge(Y_df_vol.rename(columns={'y':'vol_y'}),on=['unique_id','ds'],how='inner')

In [17]:
Y_df.unique_id.unique()

array(['3m', 'ab_inbev', 'abbvie', 'adidas', 'adobe', 'aia',
       'air_liquide', 'airbus', 'alibaba', 'allianz', 'alupar', 'amazon',
       'ambev', 'amd', 'american_express', 'amgen', 'anglo_american',
       'apple', 'arcelormittal', 'asml', 'assai', 'astrazeneca', 'att',
       'aura', 'axa', 'b3', 'baidu', 'banco_brasil', 'bank_of_america',
       'barclays', 'basf', 'bb_seguridade', 'berkshire', 'bhp',
       'blackrock', 'bnp_paribas', 'boeing', 'booking', 'bp', 'bradesco',
       'bradespar', 'broadcom', 'btg_pactual', 'byd', 'caterpillar',
       'cemig', 'chevron', 'china_construction_bank', 'cisco',
       'citigroup', 'coca_cola', 'colgate', 'comcast',
       'commonwealth_bank', 'conocophillips', 'cosan', 'costco', 'cpfl',
       'crowdstrike', 'csl', 'csn', 'csn_mineracao', 'cvc', 'cyrela',
       'danone', 'dasa', 'dbs', 'deere', 'deutsche_bank',
       'deutsche_telekom', 'diageo', 'disney', 'duke_energy',
       'ecorodovias', 'eli_lilly', 'enel', 'energisa', 'engie_b

In [18]:
mode='validacao'
horizon=60
Y_df_train_15, Y_df_valid_15, Y_df_test_15= split_data(Y_df,15,mode)
Y_df_train_30, Y_df_valid_30, Y_df_test_30= split_data(Y_df,30,mode)
Y_df_train_45, Y_df_valid_45, Y_df_test_45= split_data(Y_df,45,mode)
Y_df_train_60, Y_df_valid_60, Y_df_test_60= split_data(Y_df,60,mode)
Y_df_train_90, Y_df_valid_90, Y_df_test_90= split_data(Y_df,90,mode)
Y_df_train_180, Y_df_valid_180, Y_df_test_180= split_data(Y_df,180,mode)

In [19]:
Y_df_train_15.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_15.parquet') 
Y_df_valid_15.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_15.parquet')  
Y_df_test_15.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_15.parquet') 
Y_df_train_30.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_30.parquet') 
Y_df_valid_30.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_30.parquet')  
Y_df_test_30.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_30.parquet') 
Y_df_train_45.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_45.parquet')
Y_df_valid_45.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_45.parquet')
Y_df_test_45.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_45.parquet')
Y_df_train_60.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_60.parquet')
Y_df_valid_60.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_60.parquet')
Y_df_test_60.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_60.parquet')
Y_df_train_90.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_90.parquet')
Y_df_valid_90.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_90.parquet')
Y_df_test_90.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_90.parquet')
Y_df_train_180.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_180.parquet')
Y_df_valid_180.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_180.parquet') 
Y_df_test_180.to_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_180.parquet')

In [20]:
Y_df_train_transformed_15, Y_df_valid_future_15 = finance_feature_transform(Y_df_train_15, 'D' ,15,market="both",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True)

In [21]:
Y_df_train_transformed_30, Y_df_valid_future_30 = finance_feature_transform(Y_df_train_30, 'D' ,30,market="both",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True)

In [22]:
Y_df_train_transformed_45, Y_df_valid_future_45 = finance_feature_transform(Y_df_train_45, 'D' ,45,market="both",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True)

In [23]:
Y_df_train_transformed_60, Y_df_valid_future_60 = finance_feature_transform(Y_df_train_60, 'D' ,60,market="both",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True)

In [24]:
Y_df_train_transformed_90, Y_df_valid_future_90 = finance_feature_transform(Y_df_train_90, 'D' ,90,market="both",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True)

In [25]:
Y_df_train_transformed_180, Y_df_valid_future_180 = finance_feature_transform(Y_df_train_180, 'D' ,180,market="both",
    copom_dates=None,
    fomc_dates=None,
    include_trend=True)

## Declaração dos Modelos

In [26]:
import torch.nn.functional as F
from neuralforecast.losses.pytorch import BasePointLoss


class TweedieLoss(BasePointLoss):
    """
    Deviância de Tweedie para 1 < p < 2 (regime Poisson-Gamma composto).

    Apropriada para targets contínuos não-negativos com massa em zero:
    sinistros de seguro, demanda de varejo, precipitação.

    Deviância unitária:
        d(y, μ) = 2 [ y^{2-p}/((1-p)(2-p))
                      - y μ^{1-p}/(1-p)
                      + μ^{2-p}/(2-p) ]
    válida para y ≥ 0, μ > 0.
    """
    def __init__(self, p: float = 1.5, horizon_weight=None):
        assert 1.0 < p < 2.0, "p deve estar em (1, 2)"
        super().__init__(
            horizon_weight=horizon_weight,
            outputsize_multiplier=1,
            output_names=[""],
        )
        self.p = p

    def domain_map(self, y_hat: torch.Tensor) -> torch.Tensor:
        # μ > 0 obrigatório; softplus é mais estável que exp para valores grandes
        return F.softplus(y_hat).squeeze(-1)

    def __call__(self, y, y_hat, mask=None):
        mask = self._compute_weights(y=y, mask=mask)
        p, eps = self.p, 1e-8

        mu = torch.clamp(y_hat, min=eps)
        y  = torch.clamp(y, min=0.0)  # y < 0 não pertence ao suporte

        # Em y=0: term_y = 0 (pois 2-p > 0) e term_xy = 0; sobra μ^{2-p}/(2-p).
        term_y  = torch.pow(y, 2 - p) / ((1 - p) * (2 - p))
        term_xy = y * torch.pow(mu, 1 - p) / (1 - p)
        term_mu = torch.pow(mu, 2 - p) / (2 - p)
        deviance = 2.0 * (term_y - term_xy + term_mu)

        return (deviance * mask).sum() / (mask.sum() + eps)

In [ ]:
import optuna
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import DistributionLoss, MQLoss, HuberLoss

H = 15
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]

# ---------- catálogo de losses (FUNÇÃO DE PERDA DE TREINO) -------------------

LOSSES = {
    "normal":  DistributionLoss(distribution="Normal",
                                quantiles=QUANTILES, return_params=False),
   # "gamma":   DistributionLoss(distribution="Gamma",
    #                            quantiles=QUANTILES, return_params=False),
   # "weibull": DistributionLoss(distribution="Weibull",
    #                            quantiles=QUANTILES, return_params=False),
    "mq":      MQLoss(quantiles=QUANTILES),
    "huber":   HuberLoss(delta=1.0),
}

# Cada loss é distribuição ou ponto.
IS_DISTRIBUTION = {
    "normal":  True,
    "gamma":   True,
    "weibull": True,
    "mq":      False,
    "huber":   False,
}

# Gamma e Weibull exigem y > 0 — sem zeros, sem negativos.
REQUIRES_POSITIVE = {"gamma", "weibull"}


# ---------- helpers de config (espaço de busca por trial) -------------------

def _base_train(trial):
    return {
        "hist_exog_list":['vol_y'], 
        "futr_exog_list" : Y_df_train_transformed_90.drop(['y','unique_id','ds','vol_y'],axis=1).columns.values,
        "max_steps":     trial.suggest_categorical("max_steps", [5000, 10000, 20000]),
        "batch_size":    trial.suggest_categorical("batch_size", [32, 64, 128]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "scaler_type":   trial.suggest_categorical("scaler_type",
                                                   ["robust", "standard"]),  # "identity" diverge com exógenas de grande magnitude
        "random_seed":   trial.suggest_int("random_seed", 1, 10_000),
        "early_stop_patience_steps": 10,
        "val_check_steps":           50,
    }

def _input_size(trial, mults=(1, 2, 3, 5, 7)):
    return trial.suggest_categorical("input_size", [m * H for m in mults])


def nhits_cfg(trial):
    return {**_base_train(trial),
            "input_size":         _input_size(trial),
            "n_pool_kernel_size": trial.suggest_categorical(
                "n_pool_kernel_size",
                [[2,2,2], [4,4,4], [8,4,1], [16,8,1]]),
            "n_freq_downsample":  trial.suggest_categorical(
                "n_freq_downsample",
                [[168,24,1], [24,12,1], [1,1,1]]),
            "mlp_units":          trial.suggest_categorical(
                "mlp_units", [3*[[256,256]], 3*[[512,512]]]),
            "dropout_prob_theta": trial.suggest_float("dropout_prob_theta", 0.0, 0.3)}

def nbeatsx_cfg(trial):
    return {**_base_train(trial),
            "input_size":  _input_size(trial),
            "n_blocks":    trial.suggest_categorical(
                "n_blocks", [[1,1,1], [2,2,2], [3,3,3]]),
            "mlp_units":   trial.suggest_categorical(
                "mlp_units", [3*[[256,256]], 3*[[512,512]]]),
            "stack_types": ["identity", "trend", "seasonality"]}

def tft_cfg(trial):
    return {**_base_train(trial),
            "input_size":   _input_size(trial),
            "hidden_size":  trial.suggest_categorical("hidden_size", [64, 128, 256]),
            "n_head":       trial.suggest_categorical("n_head", [2, 4, 8]),
            "dropout":      trial.suggest_float("dropout", 0.0, 0.3),
            "attn_dropout": trial.suggest_float("attn_dropout", 0.0, 0.3)}

def lstm_cfg(trial):
    return {**_base_train(trial),
            "input_size":          -1,
            "encoder_hidden_size": trial.suggest_categorical(
                "encoder_hidden_size", [64, 128, 256]),
            "encoder_n_layers":    trial.suggest_categorical(
                "encoder_n_layers", [1, 2, 3]),
            "context_size":        trial.suggest_categorical(
                "context_size", [5, 10, 20]),
            "decoder_hidden_size": trial.suggest_categorical(
                "decoder_hidden_size", [64, 128, 256])}

def patchtst_cfg(trial):
    return {**_base_train(trial),
            "input_size":     _input_size(trial),
            "hidden_size":    trial.suggest_categorical("hidden_size", [64, 128, 256]),
            "n_heads":        trial.suggest_categorical("n_heads", [4, 8, 16]),
            "patch_len":      trial.suggest_categorical("patch_len", [8, 16, 24]),
            "stride":         trial.suggest_categorical("stride", [4, 8, 16]),
            "encoder_layers": trial.suggest_categorical("encoder_layers", [2, 3, 4]),
            "dropout":        trial.suggest_float("dropout", 0.0, 0.3)}

def itransformer_cfg(trial):
    return {**_base_train(trial),
            "input_size":  _input_size(trial),
            "hidden_size": trial.suggest_categorical("hidden_size", [128, 256, 512]),
            "n_heads":     trial.suggest_categorical("n_heads", [4, 8]),
            "e_layers":    trial.suggest_categorical("e_layers", [2, 3, 4]),
            "d_ff":        trial.suggest_categorical("d_ff", [256, 512, 1024]),
            "dropout":     trial.suggest_float("dropout", 0.0, 0.3)}

def tsmixer_cfg(trial):
    return {**_base_train(trial),
            "input_size": _input_size(trial),
            "n_block":    trial.suggest_categorical("n_block", [2, 4, 6]),
            "ff_dim":     trial.suggest_categorical("ff_dim", [64, 128, 256]),
            "dropout":    trial.suggest_float("dropout", 0.0, 0.3)}

def deepar_cfg(trial):
    return {**_base_train(trial),
            "input_size":       _input_size(trial),
            "lstm_hidden_size": trial.suggest_categorical(
                "lstm_hidden_size", [64, 128, 256]),
            "lstm_n_layers":    trial.suggest_categorical(
                "lstm_n_layers", [1, 2, 3]),
            "lstm_dropout":     trial.suggest_float("lstm_dropout", 0.0, 0.3)}


# Catálogo: nome do modelo -> função de config (trial -> dict de hiperparâmetros)
MODEL_REGISTRY = {
    "NHITS":        nhits_cfg,
    "NBEATSx":      nbeatsx_cfg,
    "TFT":          tft_cfg,
    "LSTM":         lstm_cfg,
    "PatchTST":     patchtst_cfg,
    "iTransformer": itransformer_cfg,
    "TSMixer":      tsmixer_cfg,
    "DeepAR":       deepar_cfg,
}

# Orçamento de busca (nº de trials do Optuna)
NUM_TRIALS = 50


## Pipeline de Sweep de Hiperparâmetros (Optuna + MLflow)

Pipeline para **escolher o modelo** e ajustar hiperparâmetros com **Optuna**,
acompanhando ao vivo no **MLflow** um **score probabilístico** (CRPS).

Fluxo:
1. **Infraestrutura MLflow** — `setup_mlflow` e `_log_optuna_plots`.
2. **Configuração** — escolha `MODEL_NAME` e `LOSS_NAME` (de distribuição, ex.: `"normal"`).
3. **Sweep CRPS** — `run_sweep_prob` otimiza o score probabilístico, loga cada trial
   num run aninhado e gerencia a VRAM (limpeza por trial, OOM não-fatal, mixed precision).

Backend do MLflow: `sqlite:///mlflow.db`. UI: `mlflow ui --backend-store-uri sqlite:///mlflow.db`.


In [ ]:
# =============================================================================
# Infraestrutura de acompanhamento no MLflow
# =============================================================================
# Requisitos:  pip install mlflow plotly

import os
import tempfile
import optuna
import mlflow


def setup_mlflow(experiment_name, tracking_uri=None):
    """Configura o destino do tracking e o experimento ativo."""
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)

    # Fazemos o logging manualmente (um run por trial). Desligamos o autolog do
    # PyTorch Lightning para evitar runs duplicados e o warning de checkpoint.
    try:
        mlflow.autolog(disable=True)
    except Exception as e:
        print(f"[mlflow] nao foi possivel desativar o autolog: {e}")

    mlflow.set_experiment(experiment_name)
    print(f"[mlflow] tracking_uri = {mlflow.get_tracking_uri()}")
    print(f"[mlflow] experiment   = {experiment_name}")


def _log_optuna_plots(study):
    """Salva gráficos do Optuna (histórico + importância) como artefatos."""
    try:
        import optuna.visualization as vis
        with tempfile.TemporaryDirectory() as tmp:
            figs = {
                "optuna_history.html":     vis.plot_optimization_history,
                "optuna_importances.html": vis.plot_param_importances,
                "optuna_parallel.html":    vis.plot_parallel_coordinate,
            }
            for fname, fn in figs.items():
                try:
                    path = os.path.join(tmp, fname)
                    fn(study).write_html(path)
                    mlflow.log_artifact(path, artifact_path="optuna")
                except Exception as e:
                    print(f"[mlflow] gráfico {fname} ignorado: {e}")
    except Exception as e:
        print(f"[mlflow] visualizações do Optuna indisponíveis (instale plotly): {e}")


In [ ]:
# =============================================================================
# CONFIGURAÇÃO DO EXPERIMENTO  ->  escolha aqui ANTES de rodar
# =============================================================================

# 1) Modelo a otimizar (chave de MODEL_REGISTRY)
MODEL_NAME  = "NHITS"      # opções: list(MODEL_REGISTRY) -> NHITS, NBEATSx, TFT, LSTM, PatchTST, iTransformer, TSMixer, DeepAR

# 2) Função de perda de TREINO (chave de LOSSES) — precisa ser de DISTRIBUIÇÃO p/ o CRPS
LOSS_NAME   = "normal"     # opções: list(LOSSES) -> normal, mq, huber

# 3) Orçamento de busca e split de validação
NUM_TRIALS  = 50           # nº de trials do Optuna
VAL_SIZE    = H * 4        # tamanho da janela de validação usada na cross_validation
FREQ        = "D"

# 4) Onde o MLflow grava os resultados (file store './mlruns' foi descontinuado)
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"   # ou servidor remoto: "http://127.0.0.1:5000"
EXPERIMENT_NAME     = f"forecast_global__{MODEL_NAME}__{LOSS_NAME}"

print("Modelos disponíveis :", list(MODEL_REGISTRY))
print("Losses disponíveis  :", list(LOSSES))
print("-" * 60)
print(f"Selecionado -> modelo={MODEL_NAME} | loss={LOSS_NAME}")
print(f"MLflow experiment   : {EXPERIMENT_NAME}")

# Validação imediata das escolhas (falha cedo se algo estiver errado)
assert MODEL_NAME in MODEL_REGISTRY, f"MODEL_NAME inválido: {MODEL_NAME}"
assert LOSS_NAME  in LOSSES,         f"LOSS_NAME inválido: {LOSS_NAME}"


## Acompanhamento por Score Probabilístico (CRPS) Durante o Sweep

Para previsão **probabilística**, otimizar erro de ponto (MAE/RMSE) — ou mesmo a
log-verossimilhança gaussiana, que com `sigma` ~constante vira um **MSE
reescalado** — não avalia a distribuição prevista como um todo.

O score probabilístico canônico é o **CRPS (Continuous Ranked Probability
Score)**: uma regra de pontuação **própria** que generaliza o MAE para
distribuições. Para a Normal há forma fechada usando `loc`/`scale`:

$$\text{CRPS}(\mathcal{N}(\mu,\sigma),y)=\sigma\Big[z(2\Phi(z)-1)+2\varphi(z)-\tfrac{1}{\sqrt\pi}\Big],\;z=\tfrac{y-\mu}{\sigma}.$$

O sweep abaixo, a cada trial:
1. Treina com a loss de distribuição em modo `return_params=True`
   (a `cross_validation` devolve `loc`/`scale`).
2. Calcula **CRPS** (probabilístico, próprio), e também `loglik` e métricas de
   erro como referência.
3. Loga tudo ao vivo num run aninhado do MLflow.

- O Optuna **MINIMIZA** `crps` por padrão (`PROB_SCORE="crps"`); `loglik` é
  maximizada se escolhida.
- Forma fechada implementada para **Normal**; para distribuição-livre (ex.:
  `MQLoss`), use o **CRPS por quantis** = média do *pinball loss* sobre a grade
  de quantis (equivalente ao que a `MQLoss` otimiza).


In [ ]:
# =============================================================================
# Sweep otimizando SCORE PROBABILÍSTICO (CRPS), com log ao vivo + gestão de VRAM
# =============================================================================
import gc
import numpy as np
from scipy.special import erf  # scipy é dependência do neuralforecast
from neuralforecast.models import (
    NHITS, NBEATSx, TFT, LSTM, PatchTST, iTransformer, TSMixer, DeepAR,
)

BASE_MODELS = {
    "NHITS": NHITS, "NBEATSx": NBEATSx, "TFT": TFT, "LSTM": LSTM,
    "PatchTST": PatchTST, "iTransformer": iTransformer,
    "TSMixer": TSMixer, "DeepAR": DeepAR,
}

# Qual score probabilístico otimizar: "crps" (minimiza) ou "loglik" (maximiza)
PROB_SCORE = "crps"
_DIRECTION = {"crps": "minimize", "loglik": "maximize"}

# --------------------------- Controle de VRAM -------------------------------
PRECISION      = "bf16-mixed"   # mixed precision ~metade da VRAM; use "16-mixed" se a GPU não tiver bf16, ou "32-true" p/ desligar
MAX_BATCH_SIZE = 64             # teto de batch_size (limita VRAM)
MAX_INPUT_SIZE = 5 * H          # teto de input_size (janelas longas pesam muito)


def _free_gpu():
    """Libera memória de GPU acumulada entre trials."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _trainer_kwargs():
    """Kwargs do Lightning Trainer (repassados pelo modelo) p/ economizar memória."""
    acc, dev = _detectar_acelerador()
    return dict(
        accelerator=acc,
        devices=dev,
        precision=PRECISION,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        gradient_clip_val=1.0,   # evita explosão de gradiente -> NaN
    )


def _is_oom(exc):
    return isinstance(exc, getattr(torch.cuda, "OutOfMemoryError", ())) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower()
    )


def _is_divergence(exc):
    """Divergência numérica: parâmetros viraram NaN/Inf durante o treino."""
    msg = str(exc).lower()
    return isinstance(exc, ValueError) and (
        "invalid values" in msg or "satisfy the constraint" in msg or "nan" in msg
    )


def _norm_cdf(z):
    return 0.5 * (1.0 + erf(z / np.sqrt(2.0)))

def _norm_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)


def _compute_metrics(y, yhat):
    """Métricas de ERRO (referência), sobre a previsão de ponto (média)."""
    err = y - yhat
    ae = np.abs(err)
    denom = np.abs(y) + np.abs(yhat) + 1e-8
    return {
        "mae":   float(np.mean(ae)),
        "rmse":  float(np.sqrt(np.mean(err ** 2))),
        "bias":  float(np.mean(err)),
        "smape": float(np.mean(2.0 * ae / denom)),
    }


def _find_param_col(df, alias, suffix):
    cands = [c for c in df.columns if c.startswith(alias) and c.endswith(suffix)]
    if not cands:
        raise KeyError(
            f"Coluna '{suffix}' não encontrada para '{alias}' "
            f"(precisa de return_params=True). Colunas: {list(df.columns)}")
    return cands[0]


def _make_dist_loss_with_params(loss_name):
    """Recria a loss de distribuição escolhida com return_params=True."""
    if not IS_DISTRIBUTION.get(loss_name, False):
        raise ValueError(
            f"Score probabilístico paramétrico requer loss de distribuição; "
            f"'{loss_name}' não é. Use LOSS_NAME='normal' (ou veja CRPS por quantis).")
    base = LOSSES[loss_name]
    distribution = getattr(base, "distribution", None) or "Normal"
    return DistributionLoss(distribution=distribution,
                            quantiles=QUANTILES, return_params=True), distribution


def _prob_scores_normal(y, loc, scale):
    """CRPS (forma fechada) e log-verossimilhança média sob a Normal prevista."""
    scale = np.clip(scale, 1e-6, None)
    z = (y - loc) / scale
    crps = scale * (z * (2.0 * _norm_cdf(z) - 1.0)
                    + 2.0 * _norm_pdf(z) - 1.0 / np.sqrt(np.pi))
    loglik = -0.5 * np.log(2 * np.pi) - np.log(scale) - 0.5 * z ** 2
    return {"crps": float(np.mean(crps)), "loglik": float(np.mean(loglik))}


def _crps_from_quantiles(y, q_preds, quantiles):
    """CRPS distribution-free = 2 * média do pinball loss sobre a grade de quantis.

    `q_preds`: array (n_obs, n_quantis); `quantiles`: lista de níveis em (0,1).
    Útil quando não há parâmetros (ex.: MQLoss).
    """
    y = y[:, None]
    q = np.asarray(quantiles)[None, :]
    e = y - q_preds
    pinball = np.maximum(q * e, (q - 1.0) * e)   # quantile/pinball loss
    return float(2.0 * np.mean(pinball))


def _cap_for_vram(cfg):
    """Limita os hiperparâmetros que mais consomem VRAM (sem quebrar o trial)."""
    if cfg.get("batch_size") and cfg["batch_size"] > MAX_BATCH_SIZE:
        cfg["batch_size"] = MAX_BATCH_SIZE
    if isinstance(cfg.get("input_size"), int) and cfg["input_size"] > MAX_INPUT_SIZE:
        cfg["input_size"] = MAX_INPUT_SIZE
    return cfg


def run_sweep_prob(Y_df, model_name, loss_name, prob_score=PROB_SCORE,
                   h=H, num_trials=NUM_TRIALS, val_size=None, freq="D",
                   n_windows=1, experiment_name=None, tracking_uri=None):
    """Sweep que OTIMIZA um score probabilístico (CRPS/loglik), com log ao vivo.

    Robusto a OOM: libera a GPU a cada trial e, se faltar VRAM, marca o trial
    como podado (optuna.TrialPruned) em vez de derrubar o estudo inteiro.
    Retorna (study, run_id).
    """
    val_size = val_size if val_size is not None else h * 4
    experiment_name = experiment_name or f"forecast_global__{model_name}__{loss_name}__{prob_score}"
    setup_mlflow(experiment_name, tracking_uri)

    cls = BASE_MODELS[model_name]
    cfg_fn = MODEL_REGISTRY[model_name]
    loss_params, distribution = _make_dist_loss_with_params(loss_name)
    if distribution != "Normal":
        raise NotImplementedError(
            f"Forma fechada de CRPS/loglik implementada só para 'Normal' "
            f"(recebido '{distribution}'). Para outras, use _crps_from_quantiles.")

    df = Y_df
    if loss_name in REQUIRES_POSITIVE:
        n_before = len(df)
        df = df[df["y"] > 0].copy()
        if len(df) < n_before:
            print(f"[{loss_name}] filtrado: {n_before - len(df)} linhas com y<=0")

    sampler = optuna.samplers.TPESampler(seed=42, multivariate=True, group=True)
    study = optuna.create_study(direction=_DIRECTION[prob_score], sampler=sampler,
                                study_name=experiment_name)

    def objective(trial):
        cfg = _cap_for_vram(cfg_fn(trial))
        nf = None
        try:
            model = cls(h=h, loss=loss_params, alias=model_name,
                        **cfg, **_trainer_kwargs())
            nf = NeuralForecast(models=[model], freq=freq)
            cv = nf.cross_validation(df=df, n_windows=n_windows, val_size=val_size)

            y = cv["y"].to_numpy()
            loc = cv[_find_param_col(cv, model_name, "-loc")].to_numpy()
            scale = cv[_find_param_col(cv, model_name, "-scale")].to_numpy()

            metrics = _prob_scores_normal(y, loc, scale)     # crps + loglik
            metrics.update(_compute_metrics(y, loc))         # erros (média = loc), referência

            with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
                mlflow.log_params(trial.params)
                mlflow.set_tag("trial_number", trial.number)
                for m, v in metrics.items():
                    mlflow.log_metric(m, v)

            for m, v in metrics.items():
                trial.set_user_attr(m, v)
            return metrics[prob_score]

        except Exception as e:                # noqa: BLE001 - precisamos inspecionar OOM
            if _is_oom(e):
                print(f"[trial {trial.number}] OOM (batch={cfg.get('batch_size')}, "
                      f"input={cfg.get('input_size')}) — podando e seguindo.")
                raise optuna.TrialPruned()
            if _is_divergence(e):
                print(f"[trial {trial.number}] divergência numérica (NaN/Inf) "
                      f"(lr={cfg.get('learning_rate'):.2e}, scaler={cfg.get('scaler_type')}) "
                      f"— podando e seguindo.")
                raise optuna.TrialPruned()
            raise
        finally:
            del nf
            _free_gpu()                       # libera VRAM SEMPRE, entre trials

    with mlflow.start_run(run_name=f"{model_name}__{loss_name}__{prob_score}") as parent:
        mlflow.set_tags({"model": model_name, "loss": loss_name,
                         "distribution": distribution,
                         "optimize_metric": prob_score, "mode": "prob_score",
                         "precision": PRECISION})
        mlflow.log_params({"h": h, "num_trials": num_trials,
                           "val_size": val_size, "n_windows": n_windows, "freq": freq,
                           "precision": PRECISION,
                           "max_batch_size": MAX_BATCH_SIZE,
                           "max_input_size": MAX_INPUT_SIZE})
        study.optimize(objective, n_trials=num_trials)

        completed = [t for t in study.trials if t.value is not None]
        if not completed:
            print("[aviso] nenhum trial concluiu (todos OOM?). "
                  "Reduza MAX_BATCH_SIZE/MAX_INPUT_SIZE ou use precision menor.")
            return study, parent.info.run_id

        best = study.best_trial
        mlflow.log_params({f"best__{k}": v for k, v in best.params.items()})
        for m, v in best.user_attrs.items():
            mlflow.log_metric(f"best_{m}", float(v))
        mlflow.set_tag("best_trial_number", best.number)
        _log_optuna_plots(study)
        run_id = parent.info.run_id

    print(f"\nMelhor trial #{best.number} | {prob_score}={best.value:.6f}")
    print("Scores do melhor trial:", best.user_attrs)
    print("MLflow run_id (pai):", run_id)
    return study, run_id


# ----------------------- execução -------------------------------------------
# Requer LOSS_NAME de distribuição (ex.: "normal").
study_prob, run_id_prob = run_sweep_prob(
    Y_df=Y_df_train_transformed_15,
    model_name=MODEL_NAME,
    loss_name=LOSS_NAME,
    prob_score=PROB_SCORE,     # "crps" (probabilístico, próprio) ou "loglik"
    h=H,
    num_trials=NUM_TRIALS,
    val_size=VAL_SIZE,
    freq=FREQ,
    experiment_name=EXPERIMENT_NAME + f"__{PROB_SCORE}",
    tracking_uri=MLFLOW_TRACKING_URI,
)


## Avaliação no Teste e Registro do Melhor Modelo

Após o sweep retornar `study_prob`, o melhor modelo é:
1. **Reconstruído** com os melhores hiperparâmetros (`optuna.trial.FixedTrial(study.best_params)`
   alimentando a mesma função de config).
2. **Re-treinado** no conjunto de treino.
3. **Avaliado** com TODAS as métricas no conjunto de **teste** — probabilísticas
   (`crps`, `loglik`), de erro (`mae`, `rmse`, `bias`, `smape`) e de calibração
   (`coverage_50`, `coverage_80`).
4. **Registrado** no MLflow Model Registry (flavor `pyfunc` envolvendo o `NeuralForecast`),
   com as métricas de teste e as previsões logadas no run.


In [ ]:
# =============================================================================
# Avaliação no TESTE + registro do melhor modelo no MLflow Model Registry
# =============================================================================
import os
import tempfile
import mlflow.pyfunc

REGISTERED_MODEL_NAME = f"{MODEL_NAME}_{LOSS_NAME}_forecast"


def rebuild_best_model(study, model_name, loss_name, h=H):
    """Reconstrói o modelo com os MELHORES hiperparâmetros do estudo."""
    cfg_fn = MODEL_REGISTRY[model_name]
    cfg = _cap_for_vram(cfg_fn(optuna.trial.FixedTrial(study.best_params)))
    loss_params, distribution = _make_dist_loss_with_params(loss_name)
    cls = BASE_MODELS[model_name]
    model = cls(h=h, loss=loss_params, alias=model_name, **cfg, **_trainer_kwargs())
    return model, distribution


def _coverage(df, alias, level):
    lo, hi = f"{alias}-lo-{level:.1f}", f"{alias}-hi-{level:.1f}"
    if lo in df.columns and hi in df.columns:
        return float(((df["y"] >= df[lo]) & (df["y"] <= df[hi])).mean())
    return None


def evaluate_all_metrics(pred_df, y_true_df, alias, distribution):
    """Junta previsão x teste e calcula todas as métricas."""
    m = pred_df.merge(y_true_df[["unique_id", "ds", "y"]],
                      on=["unique_id", "ds"], how="inner")
    if m.empty:
        raise ValueError("Merge previsão x teste vazio: verifique datas/unique_id.")
    y = m["y"].to_numpy()
    metrics = {}
    if distribution == "Normal":                              # probabilísticos
        loc = m[_find_param_col(m, alias, "-loc")].to_numpy()
        scale = m[_find_param_col(m, alias, "-scale")].to_numpy()
        metrics.update(_prob_scores_normal(y, loc, scale))    # crps, loglik
    if alias in m.columns:                                    # erro de ponto (média)
        metrics.update(_compute_metrics(y, m[alias].to_numpy()))
    for lvl in (50.0, 80.0):                                  # calibração
        cov = _coverage(m, alias, lvl)
        if cov is not None:
            metrics[f"coverage_{int(lvl)}"] = cov
    return metrics, m


class NeuralForecastModel(mlflow.pyfunc.PythonModel):
    """Wrapper pyfunc: carrega o NeuralForecast salvo e prevê a partir de futr_df."""
    def load_context(self, context):
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(path=context.artifacts["nf_dir"])

    def predict(self, context, model_input):
        return self.nf.predict(futr_df=model_input)


def _log_pyfunc_model(**kwargs):
    try:
        return mlflow.pyfunc.log_model(name="model", **kwargs)            # MLflow >= 3
    except TypeError:
        return mlflow.pyfunc.log_model(artifact_path="model", **kwargs)   # MLflow 2.x


def _log_metrics_safe(metrics, prefix=""):
    """Loga métricas no MLflow ignorando valores não-finitos (NaN/Inf).

    Motivo: o backend SQLite do MLflow guarda NaN como is_nan=1 e, ao registrar
    o modelo na MESMA transação (autoflush), pode colidir na UNIQUE constraint
    da tabela `metrics` (key, timestamp, step, value, is_nan, run_uuid) ->
    'IntegrityError: UNIQUE constraint failed'. Além disso, métrica NaN não
    carrega informação. Então logamos só as finitas e sinalizamos as demais
    como tag, sem derrubar a run.
    """
    finite, nonfinite = {}, []
    for k, v in metrics.items():
        key = f"{prefix}{k}"
        if v is not None and np.isfinite(v):
            finite[key] = float(v)
        else:
            nonfinite.append(key)
    if finite:
        mlflow.log_metrics(finite)
    if nonfinite:
        mlflow.set_tag("metrics_nan", ",".join(nonfinite))
        print(f"[aviso] métricas não-finitas (NaN/Inf) NÃO logadas: {nonfinite}")
    return finite, nonfinite


def finalize_best_model(study, model_name, loss_name, train_df, futr_df, y_true_df,
                        h=H, val_size=None, freq="D",
                        experiment_name=None, tracking_uri=None,
                        registered_model_name=None):
    """Refit do melhor modelo -> avalia TODAS as métricas no teste -> registra no MLflow.

    Retorna (nf, pred_df, test_metrics, model_uri).
    """
    val_size = val_size if val_size is not None else h * 4
    experiment_name = experiment_name or EXPERIMENT_NAME
    registered_model_name = registered_model_name or REGISTERED_MODEL_NAME
    setup_mlflow(experiment_name, tracking_uri)

    model, distribution = rebuild_best_model(study, model_name, loss_name, h=h)
    nf = NeuralForecast(models=[model], freq=freq)

    with mlflow.start_run(run_name=f"{model_name}__{loss_name}__BEST_test") as run:
        mlflow.set_tags({"model": model_name, "loss": loss_name,
                         "distribution": distribution, "stage": "test_evaluation",
                         "best_trial_number": study.best_trial.number})
        mlflow.log_params({f"best__{k}": v for k, v in study.best_params.items()})

        try:
            nf.fit(df=train_df, val_size=val_size)
            pred = nf.predict(futr_df=futr_df)
        finally:
            _free_gpu()

        num = pred.select_dtypes("number").to_numpy()                  # diagnóstico de NaN
        n_bad = int((~np.isfinite(num)).sum()) if num.size else 0
        if n_bad:
            print(f"[aviso] previsão contém {n_bad} valores não-finitos (NaN/Inf) — "
                  f"o modelo provavelmente DIVERGIU no refit final. As métricas "
                  f"resultantes serão NaN; revise lr/scaler ou aumente o gradient_clip.")

        test_metrics, merged = evaluate_all_metrics(pred, y_true_df, model_name, distribution)
        _log_metrics_safe(test_metrics, prefix="test_")                 # ignora NaN/Inf

        with tempfile.TemporaryDirectory() as tmp:
            csv_path = os.path.join(tmp, "test_predictions.csv")    # artefato: previsões + y
            merged.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path)

            nf_dir = os.path.join(tmp, "nf_best")                   # registra no Model Registry
            nf.save(path=nf_dir, overwrite=True, save_dataset=True)
            info = _log_pyfunc_model(python_model=NeuralForecastModel(),
                                     artifacts={"nf_dir": nf_dir},
                                     registered_model_name=registered_model_name)
        model_uri = info.model_uri
        mlflow.set_tag("registered_model", registered_model_name)

    print("\n=== Métricas no TESTE ===")
    for k, v in test_metrics.items():
        print(f"  {k:12s}: {v:.6f}")
    print(f"\nModelo registrado: {registered_model_name}")
    print(f"model_uri        : {model_uri}")
    return nf, pred, test_metrics, model_uri


# ----------------------- execução -------------------------------------------
nf_best, pred_test, test_metrics, model_uri = finalize_best_model(
    study=study_prob,                       # estudo retornado pelo sweep (run_sweep_prob)
    model_name=MODEL_NAME,
    loss_name=LOSS_NAME,
    train_df=Y_df_train_transformed_15,     # treino (transformado)
    futr_df=Y_df_valid_future_15,           # exógenas futuras p/ prever o horizonte
    y_true_df=pd.concat([Y_df_valid_15, Y_df_test_15]),  # alvo REAL do teste
    h=H,
    val_size=VAL_SIZE,
    freq=FREQ,
    experiment_name=EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
    registered_model_name=REGISTERED_MODEL_NAME,
)


## Acompanhar o Treino de Outro Computador na Rede (MLflow Server)

O `mlflow ui` serve só na máquina local. Para outra máquina da **mesma rede** ver
o treino ao vivo, rode um **MLflow server** ouvindo em `0.0.0.0` e faça o notebook
logar nesse servidor.

**1) Na máquina que treina — inicie o servidor (terminal):**
```bash
mlflow server \
  --backend-store-uri sqlite:///mlflow.db \
  --artifacts-destination ./mlartifacts \
  --serve-artifacts \
  --host 0.0.0.0 --port 5000
```
- `--host 0.0.0.0` expõe na rede (não só localhost).
- `--serve-artifacts` faz o próprio servidor entregar os artefatos (gráficos do
  Optuna), então a máquina remota os enxerga sem acesso ao disco.

**2) No notebook — aponte o tracking para o servidor (antes de rodar o sweep):**
```python
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"   # treino loga via servidor (escritor único do sqlite)
```
> Logar **através** do servidor (e não direto no `sqlite:///mlflow.db`) evita
> conflito de escrita concorrente no SQLite.

**3) Descubra o IP da máquina de treino na rede local:**
```bash
hostname -I        # Linux  -> ex.: 192.168.0.42
# Windows: ipconfig | Mac: ipconfig getifaddr en0
```

**4) No outro computador — abra no navegador:**
```
http://192.168.0.42:5000
```
A UI atualiza conforme cada trial termina (clique em "Refresh"); compare runs e
ative as colunas de métricas (`crps`, `loglik`, ...).

**Observações**
- **Firewall:** libere a porta TCP **5000** na máquina de treino
  (Linux: `sudo ufw allow 5000/tcp`; Windows: regra de entrada no Firewall).
- **Mesma rede/sub-rede:** as máquinas precisam se enxergar (LAN/VPN). Para internet
  pública, prefira um túnel (ex.: `ssh -L 5000:localhost:5000 user@host`, ou
  ngrok/cloudflared) em vez de expor a porta.
- **Acesso remoto também para logar:** se o treino rodar em OUTRO host que não o do
  servidor, use `MLFLOW_TRACKING_URI="http://<IP-do-servidor>:5000"` nesse host.


## Gráficos de Acompanhamento do Treino (curvas de aprendizado)

Para inspecionar a convergência **ao longo das épocas/steps**, a NeuralForecast
guarda em cada modelo treinado duas trajetórias — `train_trajectories` e
`valid_trajectories` — como listas de `(step, loss)`, populadas **independentemente
de logger**. As funções abaixo tornam isso explícito:

- `plot_training_curves(nf, alias)` — plota treino (linha cheia) e validação
  (tracejada) num só gráfico; usa escala log no eixo da loss quando todas as perdas
  são positivas (a NLL da `DistributionLoss` pode ser negativa, então cai para linear).
- `tabela_trajetoria(nf)` — devolve a mesma trajetória como `DataFrame`
  (`step`, `train_loss`, `valid_loss`) para inspeção numérica.

> A frequência dos pontos de validação é controlada por `val_check_steps`
> (definido em `_base_train`, padrão a cada 50 steps) e o treino para cedo por
> `early_stop_patience_steps`. Para curvas mais densas, reduza `val_check_steps`.


In [ ]:
# =============================================================================
# Curvas de aprendizado (treino x validação) ao longo dos steps/épocas
# =============================================================================
def plot_training_curves(nf, alias=None, figsize=(10, 5), logy=True, ax=None):
    """Plota a evolução da loss de treino e validação registradas PELO modelo.

    A NeuralForecast guarda, em cada modelo, `train_trajectories` e
    `valid_trajectories` como listas de (step, loss) — independentemente de logger.
    Use depois de treinar (ex.: no `nf_best` retornado por finalize_best_model).
    """
    model = nf.models[0] if hasattr(nf, "models") else nf
    tr = list(getattr(model, "train_trajectories", []) or [])
    va = list(getattr(model, "valid_trajectories", []) or [])
    if not tr and not va:
        print("Sem trajetórias registradas — treine o modelo antes de plotar.")
        return None

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    vals = []
    if tr:
        a = np.asarray(tr, dtype=float)
        ax.plot(a[:, 0], a[:, 1], label="treino", lw=2, color="#1f77b4")
        vals += list(a[:, 1])
    if va:
        b = np.asarray(va, dtype=float)
        ax.plot(b[:, 0], b[:, 1], label="validação", lw=2, ls="--",
                color="#d62728", marker="o", ms=3)
        vals += list(b[:, 1])

    # escala log só se TODAS as perdas forem positivas (a NLL pode ser negativa)
    if logy and vals and min(vals) > 0:
        ax.set_yscale("log")
    ax.set_xlabel("step de treino")
    ax.set_ylabel("loss")
    ax.set_title(f"Curva de aprendizado — {alias or getattr(model, 'alias', '')}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return ax


def tabela_trajetoria(nf):
    """Trajetória de treino/validação como DataFrame (step, train_loss, valid_loss)."""
    model = nf.models[0] if hasattr(nf, "models") else nf
    tr = pd.DataFrame(getattr(model, "train_trajectories", []) or [],
                      columns=["step", "train_loss"])
    va = pd.DataFrame(getattr(model, "valid_trajectories", []) or [],
                      columns=["step", "valid_loss"])
    if tr.empty and va.empty:
        return pd.DataFrame(columns=["step", "train_loss", "valid_loss"])
    return (tr.merge(va, on="step", how="outer")
              .sort_values("step").reset_index(drop=True))


# Curvas do melhor modelo (nf_best vem da seção de avaliação/registro)
plot_training_curves(nf_best, alias=MODEL_NAME)
tabela_trajetoria(nf_best).tail(10)


## Previsão com Atualização do Dataset

Em operação, novos pregões chegam todo dia. Em vez de re-treinar do zero, o modelo
já treinado projeta o próximo horizonte **alimentando o histórico atualizado**:

1. `atualizar_dataset` — anexa as observações novas ao `Y_df`, deduplicando por
   `(unique_id, ds)` e mantendo a versão mais recente.
2. `prever_com_atualizacao` — recalcula as exógenas de calendário/Fourier sobre o
   dataset atualizado e chama `nf.predict(df=..., futr_df=...)`.
   - `refit=False` (padrão): usa os pesos atuais; rápido, para previsão recorrente.
   - `refit=True`: re-treina antes de prever, quando entrou bastante dado novo.

A previsão sai sempre para os `H` passos **após a última data** do dataset atualizado.


In [ ]:
# =============================================================================
# Previsão com ATUALIZAÇÃO do dataset (re-previsão, sem ou com refit)
# =============================================================================
# Fluxo de produção: chega dado novo -> anexa ao histórico -> recalcula as
# features de calendário/Fourier -> o modelo treinado projeta o próximo horizonte H.

def atualizar_dataset(Y_df_atual, novos_dados, ordenar=True):
    """Anexa observações novas ao histórico, removendo duplicatas (unique_id, ds).

    `novos_dados`: DataFrame com as MESMAS colunas de Y_df (unique_id, ds, y, vol_y, ...).
    Em colisão de (unique_id, ds), mantém a observação MAIS NOVA (keep='last').
    """
    base  = Y_df_atual.copy()
    novos = novos_dados.copy()
    base["ds"]  = pd.to_datetime(base["ds"])
    novos["ds"] = pd.to_datetime(novos["ds"])
    out = (pd.concat([base, novos], ignore_index=True)
             .drop_duplicates(subset=["unique_id", "ds"], keep="last"))
    if ordenar:
        out = out.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    return out


def prever_com_atualizacao(nf, Y_df_atualizado, h=H, freq=FREQ, market="both",
                           refit=False, val_size=None, include_trend=True):
    """Recalcula as features sobre o dataset atualizado e projeta o horizonte H.

    - refit=False (padrão): usa os pesos já treinados; só alimenta o histórico novo
      (rápido — ideal para previsão diária recorrente).
    - refit=True: re-treina o modelo no dataset atualizado antes de prever
      (mais lento — use quando entrar bastante dado novo).

    Retorna (forecast_df, df_transformado, futr_df).
    """
    df_trans, futr_df = finance_feature_transform(
        Y_df_atualizado, freq, h, market=market,
        copom_dates=None, fomc_dates=None, include_trend=include_trend)

    if refit:
        nf.fit(df=df_trans, val_size=val_size or h * 4)
        _free_gpu()

    fcst = nf.predict(df=df_trans, futr_df=futr_df)
    return fcst, df_trans, futr_df


# ----------------------- exemplo de uso -------------------------------------
# Em produção, `novos_dados` viria de uma API/parquet recém-baixado. Aqui, para
# ilustrar, reusamos o próprio Y_df (o anexo é idempotente por (unique_id, ds)).
Y_df_atualizado = atualizar_dataset(Y_df, Y_df)
print(f"dataset atualizado: {Y_df_atualizado.shape} | "
      f"última data = {Y_df_atualizado['ds'].max():%Y-%m-%d}")

# Projeta o próximo horizonte H usando o melhor modelo já treinado (sem refit).
forecast_novo, df_trans_atualizado, futr_atualizado = prever_com_atualizacao(
    nf_best, Y_df_atualizado, h=H, freq=FREQ, market="both", refit=False)

print("\nPrevisão para o próximo horizonte a partir do dataset atualizado:")
forecast_novo.head(10)


## Deploy Local do Melhor Modelo

Três formas de servir o melhor modelo **localmente**, da mais simples à mais isolada:

1. **Em processo (`PrevisorLocal`)** — carrega o `NeuralForecast` salvo no mesmo
   kernel/script e prevê chamando um método. Zero infraestrutura; ideal para *batch*
   diário ou para embutir em outro pipeline Python.
2. **REST com FastAPI** — gera um microserviço (`servico_previsao_fastapi.py`) que
   sobe com `uvicorn` e responde em `POST /prever`. Bom para integrar com outros
   sistemas/linguagens.
3. **REST com MLflow** — como o melhor modelo já está no **Model Registry**, o
   `mlflow models serve` expõe um endpoint sem escrever serviço algum.

Todas reaproveitam o modelo salvo com `save_dataset=True`, então o histórico fica
embutido e só as **exógenas futuras** (`futr_df`) precisam ser enviadas na previsão.


In [ ]:
# =============================================================================
# Deploy LOCAL do melhor modelo — (1) EM PROCESSO (PrevisorLocal)
# =============================================================================
from pathlib import Path

DEPLOY_DIR = Path("./deploy/nf_best")     # diretório local que guarda o modelo servido


def salvar_para_deploy(nf, caminho=DEPLOY_DIR):
    """Persiste o NeuralForecast (pesos + dataset) num diretório local de deploy."""
    caminho = Path(caminho)
    caminho.mkdir(parents=True, exist_ok=True)
    nf.save(path=str(caminho), overwrite=True, save_dataset=True)
    print(f"✓ modelo salvo para deploy em: {caminho.resolve()}")
    return caminho


class PrevisorLocal:
    """Serviço de previsão EM PROCESSO: carrega o modelo salvo e projeta H passos.

    Uso:
        svc = PrevisorLocal()                            # carrega ./deploy/nf_best
        fcst = svc.prever(df=hist_df, futr_df=futr_df)   # controle total
        fcst = svc.prever_atualizado(Y_df_atualizado)    # recalcula as features sozinho
    """
    def __init__(self, caminho=DEPLOY_DIR):
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(path=str(caminho))

    def prever(self, df=None, futr_df=None):
        return self.nf.predict(df=df, futr_df=futr_df)

    def prever_atualizado(self, Y_df_atualizado, h=H, freq=FREQ, market="both"):
        fcst, _, _ = prever_com_atualizacao(self.nf, Y_df_atualizado, h=h,
                                            freq=freq, market=market, refit=False)
        return fcst


# Salva o melhor modelo (nf_best) e testa o serviço em processo
salvar_para_deploy(nf_best)
svc_local = PrevisorLocal(DEPLOY_DIR)
fcst_local = svc_local.prever_atualizado(Y_df_atualizado, h=H)
print("Previsão do serviço local (em processo):")
fcst_local.head()


In [ ]:
%%writefile servico_previsao_fastapi.py
"""Serviço REST local de previsão para o melhor modelo (FastAPI + Uvicorn).

Subir:   uvicorn servico_previsao_fastapi:app --host 0.0.0.0 --port 8000
Health:  curl http://127.0.0.1:8000/health

O modelo foi salvo com `save_dataset=True`, então a previsão usa o histórico
embutido; o cliente envia apenas o `futr_df` (exógenas FUTURAS de calendário/Fourier).
"""
from pathlib import Path

import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
from neuralforecast import NeuralForecast

DEPLOY_DIR = Path("./deploy/nf_best")

app = FastAPI(title="Forecast Global - Melhor Modelo")
nf = NeuralForecast.load(path=str(DEPLOY_DIR))     # carregado uma vez no startup


class FutrPayload(BaseModel):
    futr: list[dict]      # exógenas futuras: unique_id, ds + colunas de calendário/Fourier


@app.get("/health")
def health():
    return {"status": "ok", "modelos": [m.alias for m in nf.models]}


@app.post("/prever")
def prever(payload: FutrPayload):
    futr_df = pd.DataFrame(payload.futr)
    futr_df["ds"] = pd.to_datetime(futr_df["ds"])
    fcst = nf.predict(futr_df=futr_df)
    return fcst.assign(ds=fcst["ds"].astype(str)).to_dict(orient="records")


### Subir e consultar o serviço REST

**Opção A — FastAPI (arquivo `servico_previsao_fastapi.py` gerado na célula acima):**
```bash
pip install fastapi uvicorn requests
uvicorn servico_previsao_fastapi:app --host 0.0.0.0 --port 8000
# checagem de saúde:
curl http://127.0.0.1:8000/health
```

**Opção B — MLflow Model Registry (sem escrever um serviço):** o melhor modelo já
foi registrado (`REGISTERED_MODEL_NAME`) na seção de avaliação. Sirva direto:
```bash
mlflow models serve -m "models:/<REGISTERED_MODEL_NAME>/latest" \
  --host 0.0.0.0 --port 8001 --env-manager local
```

**Contrato do POST** (`/prever` no FastAPI, `/invocations` no MLflow): o corpo é o
**`futr_df`** — as exógenas FUTURAS do horizonte (calendário/Fourier), geradas por
`prever_com_atualizacao(...)` / `finance_feature_transform(...)`. O histórico fica
embutido no modelo (salvo com `save_dataset=True`), então não precisa ser reenviado.

> Para expor o serviço a **outra máquina da rede**, use `--host 0.0.0.0` e libere a
> porta no firewall (mesma lógica do MLflow server da seção anterior). Em produção,
> prefira um túnel/reverse-proxy a expor a porta publicamente.


In [ ]:
# Cliente REST de exemplo (suba o uvicorn em outro terminal antes de rodar).
import json
import requests

# `futr_atualizado` vem da seção "Previsão com Atualização do Dataset":
# são as exógenas FUTURAS (calendário/Fourier) do próximo horizonte H.
_futr_json = json.loads(
    futr_atualizado.assign(ds=futr_atualizado["ds"].astype(str)).to_json(orient="records")
)
payload = {"futr": _futr_json}

try:
    r = requests.post("http://127.0.0.1:8000/prever", json=payload, timeout=60)
    print("status:", r.status_code)
    print(pd.DataFrame(r.json()).head())
except Exception as e:
    print("Serviço REST indisponível — suba-o com uvicorn em outro terminal.")
    print("Detalhe:", e)
